# Week 3 Cleaning Notebook (Processed Dataset V1)

This notebook turns EDA findings into reproducible cleaning outputs for the project pipeline.

## Scope and Week 3 Alignment

This notebook focuses on Week 3 requirements from the semester brief:
- produce a cleaned Processed Dataset V1
- preserve reproducibility and explicit table scope
- output artifacts that support schema draft and data dictionary updates

In-scope now: ratings, movies, tags, links.
Deferred to Week 5+: genome_scores and genome_tags.

In [2]:
from pathlib import Path
import json
import polars as pl

RAW_DIR = Path('../data/raw/ml-25m')
PROCESSED_DIR = Path('../data/processed/week03_v1')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    'movies.csv',
    'ratings.csv',
    'tags.csv',
    'links.csv',
]

missing_files = [name for name in required_files if not (RAW_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required raw files: {missing_files}')

RAW_DIR, PROCESSED_DIR

(PosixPath('../data/raw/ml-25m'), PosixPath('../data/processed/week03_v1'))

In [3]:
raw_profile = pl.DataFrame([
    {
        'table': 'ratings',
        'rows': pl.scan_csv(RAW_DIR / 'ratings.csv').select(pl.len()).collect().item(),
        'size_mb': round((RAW_DIR / 'ratings.csv').stat().st_size / (1024**2), 2),
    },
    {
        'table': 'movies',
        'rows': pl.scan_csv(RAW_DIR / 'movies.csv').select(pl.len()).collect().item(),
        'size_mb': round((RAW_DIR / 'movies.csv').stat().st_size / (1024**2), 2),
    },
    {
        'table': 'tags',
        'rows': pl.scan_csv(RAW_DIR / 'tags.csv').select(pl.len()).collect().item(),
        'size_mb': round((RAW_DIR / 'tags.csv').stat().st_size / (1024**2), 2),
    },
    {
        'table': 'links',
        'rows': pl.scan_csv(RAW_DIR / 'links.csv').select(pl.len()).collect().item(),
        'size_mb': round((RAW_DIR / 'links.csv').stat().st_size / (1024**2), 2),
    },
])

raw_profile.sort('rows', descending=True)

table,rows,size_mb
str,i64,f64
"""ratings""",25000095,646.84
"""tags""",1093360,37.01
"""movies""",62423,2.9
"""links""",62423,1.31


## Cleaning Rules from EDA

Applied rules:
- Keep ratings only if required fields are present and rating is in [0.5, 5.0].
- Keep movies with non-null movieId/title and preserve original genres.
- Merge links into movies so imdbId/tmdbId are directly available in the catalog table.
- Keep tmdbId nulls (observed in EDA) and document them.
- Keep tags with non-empty text and standardize whitespace.
- Add timestamp-derived datetime columns for analysis readiness.
- Export processed tables and a cleaning report to make the pipeline auditable.

In [4]:
def profile_columns(df: pl.DataFrame, table_name: str) -> pl.DataFrame:
    rows = df.height
    return pl.DataFrame([
        {
            'table': table_name,
            'column': col,
            'dtype': str(dtype),
            'null_count': int(df.select(pl.col(col).is_null().sum()).item()),
            'null_pct': round((df.select(pl.col(col).is_null().sum()).item() / rows) * 100, 4) if rows else 0.0,
        }
        for col, dtype in df.schema.items()
    ])

def to_int64_nullable(col_name: str) -> pl.Expr:
    return pl.col(col_name).cast(pl.Int64, strict=False)

def to_float64(col_name: str) -> pl.Expr:
    return pl.col(col_name).cast(pl.Float64, strict=False)

In [5]:
movies_raw = pl.read_csv(RAW_DIR / 'movies.csv', infer_schema_length=10000)
links_raw = pl.read_csv(RAW_DIR / 'links.csv', infer_schema_length=10000)

movies_clean = (
    movies_raw
    .select(['movieId', 'title', 'genres'])
    .with_columns([
        to_int64_nullable('movieId'),
        pl.col('title').cast(pl.String, strict=False),
        pl.col('genres').cast(pl.String, strict=False),
    ])
    .filter(pl.col('movieId').is_not_null() & pl.col('title').is_not_null())
    .join(
        links_raw.select(['movieId', 'imdbId', 'tmdbId']).with_columns([
            to_int64_nullable('movieId'),
            to_int64_nullable('imdbId'),
            to_int64_nullable('tmdbId'),
        ]),
        on='movieId',
        how='left',
    )
    .with_columns([
        pl.col('title').str.extract(r'\((\d{4})\)$', 1).cast(pl.Int64, strict=False).alias('release_year'),
        pl.when(pl.col('genres') == '(no genres listed)')
          .then(None)
          .otherwise(pl.col('genres'))
          .alias('genres_raw'),
    ])
    .with_columns([
        pl.col('genres_raw').str.split('|').alias('genres_list'),
        pl.when(pl.col('imdbId').is_not_null())
          .then(pl.format('tt{}', pl.col('imdbId')))
          .otherwise(None)
          .alias('imdb_title_id'),
    ])
    .sort('movieId')
)

dup_movie_ids = int(movies_clean.select(pl.col('movieId').is_duplicated().sum()).item())
if dup_movie_ids != 0:
    raise ValueError(f'movies_clean has duplicate movieId values: {dup_movie_ids}')

movies_clean.head()

movieId,title,genres,imdbId,tmdbId,release_year,genres_raw,genres_list,imdb_title_id
i64,str,str,i64,i64,i64,str,list[str],str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…",114709,862,1995,"""Adventure|Animation|Children|C…","[""Adventure"", ""Animation"", … ""Fantasy""]","""tt114709"""
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy""",113497,8844,1995,"""Adventure|Children|Fantasy""","[""Adventure"", ""Children"", ""Fantasy""]","""tt113497"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance""",113228,15602,1995,"""Comedy|Romance""","[""Comedy"", ""Romance""]","""tt113228"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""",114885,31357,1995,"""Comedy|Drama|Romance""","[""Comedy"", ""Drama"", ""Romance""]","""tt114885"""
5,"""Father of the Bride Part II (1…","""Comedy""",113041,11862,1995,"""Comedy""","[""Comedy""]","""tt113041"""


In [6]:
ratings_raw = pl.read_csv(RAW_DIR / 'ratings.csv', infer_schema_length=10000)

ratings_clean = (
    ratings_raw
    .select(['userId', 'movieId', 'rating', 'timestamp'])
    .with_columns([
        to_int64_nullable('userId'),
        to_int64_nullable('movieId'),
        to_float64('rating'),
        to_int64_nullable('timestamp'),
    ])
    .filter(
        pl.col('userId').is_not_null()
        & pl.col('movieId').is_not_null()
        & pl.col('rating').is_not_null()
        & pl.col('timestamp').is_not_null()
    )
    .filter((pl.col('rating') >= 0.5) & (pl.col('rating') <= 5.0))
    .with_columns(pl.from_epoch('timestamp', time_unit='s').alias('rated_at'))
    .sort(['userId', 'movieId', 'timestamp'])
)

dup_rating_pairs = int(ratings_clean.select(pl.struct(['userId', 'movieId']).is_duplicated().sum()).item())
if dup_rating_pairs != 0:
    raise ValueError(f'ratings_clean has duplicate (userId, movieId) pairs: {dup_rating_pairs}')

ratings_clean.head()

userId,movieId,rating,timestamp,rated_at
i64,i64,f64,i64,datetime[μs]
1,296,5.0,1147880044,2006-05-17 15:34:04
1,306,3.5,1147868817,2006-05-17 12:26:57
1,307,5.0,1147868828,2006-05-17 12:27:08
1,665,5.0,1147878820,2006-05-17 15:13:40
1,899,3.5,1147868510,2006-05-17 12:21:50


In [7]:
tags_raw = pl.read_csv(RAW_DIR / 'tags.csv', infer_schema_length=10000)

tags_clean = (
    tags_raw
    .select(['userId', 'movieId', 'tag', 'timestamp'])
    .with_columns([
        to_int64_nullable('userId'),
        to_int64_nullable('movieId'),
        pl.col('tag').cast(pl.String, strict=False),
        to_int64_nullable('timestamp'),
    ])
    .filter(
        pl.col('userId').is_not_null()
        & pl.col('movieId').is_not_null()
        & pl.col('tag').is_not_null()
        & pl.col('timestamp').is_not_null()
    )
    .with_columns(pl.col('tag').str.strip_chars().alias('tag'))
    .filter(pl.col('tag').str.len_chars() > 0)
    .with_columns([
        pl.col('tag').str.to_lowercase().alias('tag_normalized'),
        pl.from_epoch('timestamp', time_unit='s').alias('tagged_at'),
    ])
    .sort(['userId', 'movieId', 'timestamp'])
)

tags_clean.head()

userId,movieId,tag,timestamp,tag_normalized,tagged_at
i64,i64,str,i64,str,datetime[μs]
3,260,"""sci-fi""",1439472256,"""sci-fi""",2015-08-13 13:24:16
3,260,"""classic""",1439472355,"""classic""",2015-08-13 13:25:55
4,1732,"""dark comedy""",1573943598,"""dark comedy""",2019-11-16 22:33:18
4,1732,"""great dialogue""",1573943604,"""great dialogue""",2019-11-16 22:33:24
4,7569,"""so bad it's good""",1573943455,"""so bad it's good""",2019-11-16 22:30:55


In [8]:
movie_genres = (
    movies_clean
    .select(['movieId', 'genres_list'])
    .explode('genres_list')
    .rename({'genres_list': 'genre'})
    .drop_nulls(['genre'])
    .sort(['movieId', 'genre'])
)

movie_genres.head()

movieId,genre
i64,str
1,"""Adventure"""
1,"""Animation"""
1,"""Children"""
1,"""Comedy"""
1,"""Fantasy"""


In [9]:
join_checks = pl.DataFrame({
    'relationship': [
        'ratings.movieId in movies_clean.movieId',
        'tags.movieId in movies_clean.movieId',
    ],
    'unmatched_count': [
        int(ratings_clean.select('movieId').unique().join(movies_clean.select('movieId').unique(), on='movieId', how='anti').height),
        int(tags_clean.select('movieId').unique().join(movies_clean.select('movieId').unique(), on='movieId', how='anti').height),
    ],
})

before_after = pl.DataFrame([
    {
        'table': 'movies',
        'raw_rows': movies_raw.height,
        'clean_rows': movies_clean.height,
        'rows_removed': movies_raw.height - movies_clean.height,
    },
    {
        'table': 'ratings',
        'raw_rows': ratings_raw.height,
        'clean_rows': ratings_clean.height,
        'rows_removed': ratings_raw.height - ratings_clean.height,
    },
    {
        'table': 'tags',
        'raw_rows': tags_raw.height,
        'clean_rows': tags_clean.height,
        'rows_removed': tags_raw.height - tags_clean.height,
    },
    {
        'table': 'links_as_joined_columns',
        'raw_rows': links_raw.height,
        'clean_rows': movies_clean.select(['movieId', 'imdbId', 'tmdbId']).height,
        'rows_removed': links_raw.height - movies_clean.select(['movieId', 'imdbId', 'tmdbId']).height,
    },
])

join_checks, before_after

(shape: (2, 2)
 ┌─────────────────────────────────┬─────────────────┐
 │ relationship                    ┆ unmatched_count │
 │ ---                             ┆ ---             │
 │ str                             ┆ i64             │
 ╞═════════════════════════════════╪═════════════════╡
 │ ratings.movieId in movies_clea… ┆ 0               │
 │ tags.movieId in movies_clean.m… ┆ 0               │
 └─────────────────────────────────┴─────────────────┘,
 shape: (4, 4)
 ┌─────────────────────────┬──────────┬────────────┬──────────────┐
 │ table                   ┆ raw_rows ┆ clean_rows ┆ rows_removed │
 │ ---                     ┆ ---      ┆ ---        ┆ ---          │
 │ str                     ┆ i64      ┆ i64        ┆ i64          │
 ╞═════════════════════════╪══════════╪════════════╪══════════════╡
 │ movies                  ┆ 62423    ┆ 62423      ┆ 0            │
 │ ratings                 ┆ 25000095 ┆ 25000095   ┆ 0            │
 │ tags                    ┆ 1093360  ┆ 1093351    ┆ 

In [10]:
movies_path = PROCESSED_DIR / 'movies_catalog.parquet'
ratings_path = PROCESSED_DIR / 'ratings_clean.parquet'
tags_path = PROCESSED_DIR / 'tags_clean.parquet'
genres_path = PROCESSED_DIR / 'movie_genres.parquet'

movies_clean.write_parquet(movies_path)
ratings_clean.write_parquet(ratings_path)
tags_clean.write_parquet(tags_path)
movie_genres.write_parquet(genres_path)

processed_dictionary = pl.concat([
    profile_columns(movies_clean, 'movies_catalog'),
    profile_columns(ratings_clean, 'ratings_clean'),
    profile_columns(tags_clean, 'tags_clean'),
    profile_columns(movie_genres, 'movie_genres'),
], how='vertical')

dictionary_path = PROCESSED_DIR / 'week03_processed_dictionary_profile.csv'
processed_dictionary.sort(['table', 'column']).write_csv(dictionary_path)

cleaning_report = {
    'input_tables': raw_profile.to_dicts(),
    'row_changes': before_after.to_dicts(),
    'join_checks': join_checks.to_dicts(),
    'key_decisions': [
        'Merged links into movies_catalog for direct imdbId/tmdbId access',
        'Preserved null tmdbId values and documented them as expected source incompleteness',
        'Deferred genome tables to Week 5 for representation work',
    ],
    'output_tables': [
        str(movies_path),
        str(ratings_path),
        str(tags_path),
        str(genres_path),
        str(dictionary_path),
    ],
}

report_path = PROCESSED_DIR / 'week03_cleaning_report.json'
report_path.write_text(json.dumps(cleaning_report, indent=2), encoding='utf-8')

report_path, dictionary_path

(PosixPath('../data/processed/week03_v1/week03_cleaning_report.json'),
 PosixPath('../data/processed/week03_v1/week03_processed_dictionary_profile.csv'))

In [11]:
pl.DataFrame({
    'output_file': [
        str(movies_path),
        str(ratings_path),
        str(tags_path),
        str(genres_path),
        str(dictionary_path),
        str(report_path),
    ]
})

output_file
str
"""../data/processed/week03_v1/mo…"
"""../data/processed/week03_v1/ra…"
"""../data/processed/week03_v1/ta…"
"""../data/processed/week03_v1/mo…"
"""../data/processed/week03_v1/we…"
"""../data/processed/week03_v1/we…"


## Notes for Week 3 Submission

Use the generated files in data/processed/week03_v1 to support:
- Processed Dataset V1 proof
- Data dictionary draft for cleaned tables
- Schema updates showing merged movies + links strategy
- Reproducibility evidence for the ingestion-to-cleaning pipeline

In [13]:
!zip -r processed.zip /data/processed/week03_v1

  adding: data/processed/week03_v1/ (stored 0%)
  adding: data/processed/week03_v1/week03_processed_dictionary_profile.csv (deflated 69%)
  adding: data/processed/week03_v1/tags_clean.parquet (deflated 1%)
  adding: data/processed/week03_v1/movie_genres.parquet (deflated 18%)
  adding: data/processed/week03_v1/week03_cleaning_report.json (deflated 70%)
  adding: data/processed/week03_v1/ratings_clean.parquet (deflated 0%)
  adding: data/processed/week03_v1/movies_catalog.parquet (deflated 2%)
